# BottleSort Lab — Model Training

**Kunene · AI and industrial automation software simulation**

This notebook generates labelled synthetic bottle images, trains an actual pixel classifier, evaluates independent images, and exports the weights used by the browser simulator. It does not claim to recognise real bottle photographs.

## 1. Setup and reproducible image generator

Run in Google Colab or a Python environment containing NumPy and scikit-learn. If needed, install these dependencies with `pip install numpy scikit-learn` before running. The generator uses the same deterministic random algorithm and geometric pixel rules as `engine.js`. Class 0 is an aligned cap, class 1 is a missing cap, and class 2 is a tilted cap.

In [1]:
"""Train the reproducible synthetic cap-inspection model. Run with Python 3."""
from pathlib import Path
import json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

ROOT = Path.cwd()
(ROOT / 'project').mkdir(exist_ok=True)
W, H = 32, 48
CLASSES = ['cap_ok', 'missing_cap', 'tilted_cap']

class Random:
    """Mulberry32, shared verbatim in spirit with the browser renderer."""
    def __init__(self, seed): self.state = seed & 0xffffffff
    def next(self):
        self.state = (self.state + 0x6D2B79F5) & 0xffffffff
        t = self.state
        t = ((t ^ (t >> 15)) * (t | 1)) & 0xffffffff
        t ^= (t + (((t ^ (t >> 7)) * (t | 61)) & 0xffffffff)) & 0xffffffff
        return ((t ^ (t >> 14)) & 0xffffffff) / 4294967296

def image(label, seed, difficulty=0.0):
    r = Random(seed)
    cx = 16 + (r.next() - .5) * 3
    dy = (r.next() - .5) * 2
    width = 6.0 + r.next()
    light = .75 + r.next() * .5
    angle = (r.next() - .5) * .10 if label == 0 else (.35 + r.next() * .40) * (-1 if r.next() < .5 else 1)
    shift = 0 if label != 2 else (1.0 + r.next() * 1.0)
    # This is a controlled optical disturbance; it never exposes the class to inference.
    light *= 1 - difficulty * .65
    data = np.empty((H,W), dtype=np.float64)
    for y in range(H):
        for x in range(W):
            dx = x + .5 - cx
            yy = y + .5 - dy
            v = .10 + .018 * y/H
            body = abs(dx) < width and 19 <= yy < 43
            shoulder = 14 <= yy < 19 and abs(dx) < 3.2 + (yy-14)/5 * (width-3.2)
            neck = abs(dx) < 3.2 and 10 <= yy < 19
            if body or shoulder or neck:
                v = .46 + .10*(dx+width)/(width*2)
                if abs(dx+width*.63) < .65: v += .15
                if body and 25 <= yy < 35: v = .73
                if body and 28 <= yy < 31 and abs(dx)<3.7: v = .29
            if label != 1:
                xx = dx-shift
                yc = yy-10
                rx = xx*np.cos(angle)+yc*np.sin(angle)
                ry = -xx*np.sin(angle)+yc*np.cos(angle)
                if abs(rx)<4.5 and abs(ry)<2.8: v=.83 - .06*(int(x)%2)
            v = v*light + (r.next()-.5)*(.07+difficulty*.34)
            data[y,x]=np.clip(v,0,1)
    return data.ravel()

def dataset(n, start):
    rows, labels = [], []
    for c in range(3):
        for k in range(n):
            rows.append(image(c, start+c*100000+k, difficulty=.15*(k%4)))
            labels.append(c)
    return np.array(rows), np.array(labels)

def main():
    X, y = dataset(360, 104729)
    Xt, yt = dataset(140, 900001)
    model = LogisticRegression(C=.32, max_iter=600, random_state=17)
    model.fit(X,y)
    pred=model.predict(Xt)
    probabilities=model.predict_proba(Xt)
    report = {
        'model':'Multiclass logistic regression on 32 × 48 grayscale pixels',
        'classes':CLASSES,'seed':17,'training_samples':len(y),'test_samples':len(yt),
        'test_accuracy':float(accuracy_score(yt,pred)),
        'confusion_matrix':confusion_matrix(yt,pred,labels=[0,1,2]).tolist(),
        'classification_report':classification_report(yt,pred,target_names=CLASSES,output_dict=True,zero_division=0),
        'mean_confidence':float(probabilities.max(axis=1).mean()),
        'evaluation_scope':'Independent synthetic test images; not evidence of performance on real bottle photographs.',
        'train_seeds':'104729–105088; 204729–205088; 304729–305088',
        'test_seeds':'900001–900140; 1000001–1000140; 1100001–1100140',
    }
    export = {'width':W,'height':H,'classes':CLASSES,'coefficients':np.round(model.coef_,9).tolist(),
              'intercepts':np.round(model.intercept_,9).tolist(),'report':report}
    dist=ROOT/'dist'
    dist.mkdir(exist_ok=True)
    (dist/'model.json').write_text(json.dumps(export,separators=(',',':')))
    (ROOT/'project'/'evaluation.json').write_text(json.dumps(report,indent=2))
    print(json.dumps({k:report[k] for k in ['training_samples','test_samples','test_accuracy','confusion_matrix']},indent=2))




## 2. Train, evaluate, and export

Use 360 training images and 140 independently seeded test images for each class. Logistic regression learns from the flattened grayscale pixels; the true class is the training target and is never an inference input. The evaluation is on synthetic images only.

In [2]:
main()

{
  "training_samples": 1080,
  "test_samples": 420,
  "test_accuracy": 0.9976190476190476,
  "confusion_matrix": [
    [
      139,
      0,
      1
    ],
    [
      0,
      140,
      0
    ],
    [
      0,
      0,
      140
    ]
  ]
}


## 3. Inspect the measured results

The confusion-matrix rows are the known conditions and its columns are model predictions. This test accuracy is measured before the controller applies the confidence threshold.

In [3]:
report = json.loads((ROOT / 'project' / 'evaluation.json').read_text())
print(f"Held-out accuracy: {100*report['test_accuracy']:.4f}%")
print('Rows = actual; columns = predicted; class order =', CLASSES)
for name, row in zip(CLASSES, report['confusion_matrix']):
    print(f'{name:14s} {row}')
for name in CLASSES:
    item = report['classification_report'][name]
    print(f"{name:14s} precision={item['precision']:.4f} recall={item['recall']:.4f} F1={item['f1-score']:.4f}")
print(report['evaluation_scope'])

Held-out accuracy: 99.7619%
Rows = actual; columns = predicted; class order = ['cap_ok', 'missing_cap', 'tilted_cap']
cap_ok         [139, 0, 1]
missing_cap    [0, 140, 0]
tilted_cap     [0, 0, 140]
cap_ok         precision=1.0000 recall=0.9929 F1=0.9964
missing_cap    precision=1.0000 recall=1.0000 F1=1.0000
tilted_cap     precision=0.9929 recall=1.0000 F1=0.9964
Independent synthetic test images; not evidence of performance on real bottle photographs.


## 4. Reproduce the controller rule

The highest model probability below the chosen threshold produces review. Otherwise, class 0 is accepted and classes 1–2 are rejected. Test a missing-cap image without sending its class label to the classifier. The generator uses the label to construct the input image; inference uses only pixels and learned coefficients.

In [4]:
export = json.loads((ROOT / 'dist' / 'model.json').read_text())
pixels = image(1, 2007919, difficulty=0.15)
scores = np.asarray(export['coefficients']) @ pixels + np.asarray(export['intercepts'])
probabilities = np.exp(scores-scores.max())
probabilities /= probabilities.sum()
predicted = int(probabilities.argmax())
threshold = 0.85
route = 'review' if probabilities[predicted] < threshold else ('accept' if predicted == 0 else 'reject')
print('Input shape:', pixels.shape)
print('Predicted:', CLASSES[predicted])
print('Probabilities:', dict(zip(CLASSES, probabilities.round(6).tolist())))
print('Controller route:', route)
print('Browser weights exported to:', str(ROOT / 'dist' / 'model.json'))

Input shape: (1536,)
Predicted: missing_cap
Probabilities: {'cap_ok': 8.1e-05, 'missing_cap': 0.998795, 'tilted_cap': 0.001123}
Controller route: reject
Browser weights exported to: /workspace/sites/bottle-sort-lab/dist/model.json


## 5. Use the exported model

Download `dist/model.json` from the notebook files panel, place it in the application’s `dist` directory, and reload the simulator. Run the downloaded application using `python -m http.server 8000 --directory dist`.

Compare live performance under lighting disturbance values above the test range. High confidence does not guarantee a correct prediction. Training on synthetic geometric images is a learning exercise; a real installation needs representative photographs, a separate real test set, and hardware validation.